In [1]:
import pandas as pd
import numpy as np
import torch
from transformer_time_series_enc_dec import train_model,InformerForecaster,create_dataloaders,TrainConfig,inverse_transform,_init_weights
import plotly.express as px
import matplotlib.pyplot as plt
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
import json
from datetime import datetime
import os

def create_loss_plot(train_hist, val_hist, steps_hist, total_params=None):
    """Create a loss plot from training history"""
    df_train = pd.DataFrame({
        "Step": steps_hist,
        "Loss": train_hist,
        "Type": "Train"
    })
    if val_hist:
        val_steps, val_losses = zip(*val_hist)
        df_val = pd.DataFrame({
            "Step": val_steps,
            "Loss": val_losses,
            "Type": "Validation"
        })
        df_loss = pd.concat([df_train, df_val], ignore_index=True)
    else:
        df_loss = df_train
        
    title = "Training and Validation Loss"
    if total_params is not None:
        title += f"\nTotal Parameters: {total_params:,}"
    
    fig = px.line(df_loss, x="Step", y="Loss", color="Type", title=title)
    return fig, df_loss

def create_prediction_plots(model, val_loader, asset_idx, scaler, config, num_batches=10, save_dir=None):
    """Create prediction plots and calculate metrics for validation batches
    
    Args:
        model: The trained model
        val_loader: Validation data loader
        asset_idx: Index of the asset to predict
        scaler: Scaler used for data normalization
        config: Model configuration
        num_batches: Number of batches to visualize
        save_dir: If provided, save plots to this directory
        
    Returns:
        list: List of dictionaries containing metrics for each batch
    """
    metrics = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_batches:
                break
                
            # Unpack the batch tuple correctly
            x, timestamps = batch
            x = x.to(device)
            timestamps = timestamps.to(device)
            y_pred = model(x, timestamps)
            y_true = x[:, -model.pred_len:, asset_idx]

            # Move to CPU and convert to numpy
            y_pred = y_pred.cpu().numpy()
            y_true = y_true.cpu().numpy()

            # Inverse transform to real prices
            y_pred_price = inverse_transform(y_pred.flatten(), scaler, asset_idx, config["d_input"])
            y_true_price = inverse_transform(y_true.flatten(), scaler, asset_idx, config["d_input"])

            # Create plots
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            # Normalised scale
            axes[0].plot(y_true.flatten(), label="Normalised True")
            axes[0].plot(y_pred.flatten(), label="Normalised Predicted")
            axes[0].set_title(f"Batch {i+1} – Normalised")
            axes[0].set_xlabel("Prediction Step")
            axes[0].set_ylabel("Scaled Value")
            axes[0].legend()

            # Real-price scale
            axes[1].plot(y_true_price, label="Real True")
            axes[1].plot(y_pred_price, label="Real Predicted")
            axes[1].set_title(f"Batch {i+1} – Real Prices")
            axes[1].set_xlabel("Prediction Step")
            axes[1].set_ylabel("Price")
            axes[1].legend()

            plt.tight_layout()
            
            # Save or show the plot
            if save_dir:
                plt.savefig(os.path.join(save_dir, f'validation_batch_{i+1}.png'))
                plt.close()
            else:
                plt.show()

            # Calculate metrics
            true = y_true.flatten()
            pred = y_pred.flatten()

            # --- Jaggedness metrics ---
            def mean_abs_diff(x):
                return np.mean(np.abs(np.diff(x)))
            
            pred_jagg = mean_abs_diff(pred)
            true_jagg = mean_abs_diff(true)
            jagg_ratio = pred_jagg / (true_jagg + 1e-8)  # avoid divide by zero
            
            metrics.append({
                'batch': i+1,
                'true_std': float(np.std(true)),
                'pred_std': float(np.std(pred)),
                'mse': float(np.mean((true - pred) ** 2)),
                'mae': float(np.mean(np.abs(true - pred))),
                'true_jaggedness': float(true_jagg),
                'pred_jaggedness': float(pred_jagg),
                'jaggedness_ratio': float(jagg_ratio)
            })
            
    return metrics

def save_experiment_results(config, train_hist, val_hist, steps_hist, model, val_loader_1, asset_idx, scaler, num_batches=10):
    """Save experimental results including losses, plots, and metrics"""
    # Create experiment directory with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = f"experiments_{timestamp}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Calculate and save total learnable parameters
    total_learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Save configuration with model size
    config_with_params = config.copy()
    config_with_params['total_learnable_parameters'] = total_learnable_params
    with open(os.path.join(exp_dir, 'config.json'), 'w') as f:
        json.dump(config_with_params, f, indent=4)
    
    # Create and save loss plot and data
    fig, df_loss = create_loss_plot(train_hist, val_hist, steps_hist, total_learnable_params)
    fig.write_html(os.path.join(exp_dir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_dir, 'loss_history.csv'))
    
    # Create validation plots and get metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=num_batches,
        save_dir=exp_dir
    )
    
    # Save metrics
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_dir, 'validation_metrics.csv'), index=False)
    
    # Save model summary information
    with open(os.path.join(exp_dir, 'model_summary.txt'), 'w') as f:
        f.write(f"Total Learnable Parameters: {total_learnable_params:,}\n")
        f.write(f"\nModel Configuration:\n")
        for key, value in config_with_params.items():
            f.write(f"{key}: {value}\n")
    
    return exp_dir

In [4]:
# -----------------------------
# Load and preprocess data
# -----------------------------
csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]



In [5]:
# Define parameter grid
param_grid = {
    'd_model': [32,64],  # Model dimensions
    'distill': [False],   # Whether to use distillation
    'use_time_embedding': [False, True],  # Whether to use time embeddings
    'dropout': [0.05,0.1]  # Dropout rates
}

# Generate all possible combinations
from itertools import product

# Generate all combinations
keys = param_grid.keys()
configs = []
for values in product(*param_grid.values()):
    config_dict = dict(zip(keys, values))
    # Base configuration
    config = {
        "d_input": len(closes.columns),
        "n_heads": 4,
        "enc_layers": 3,
        "dec_layers": 2,
        "enc_len": 96,
        "guiding_len": 48,
        "pred_len": 24,
        "factor": 5,
    }
    # Update with current combination
    config.update(config_dict)
    # Set d_ff to 4x d_model
    config['d_ff'] = config['d_model'] * 4
    configs.append(config)

print(f"Total number of configurations to test: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"\nConfiguration {i+1}:")
    print(f"d_model: {cfg['d_model']}, d_ff: {cfg['d_ff']}")
    print(f"distill: {cfg['distill']}, use_time_embedding: {cfg['use_time_embedding']}")
    print(f"dropout: {cfg['dropout']}")

Total number of configurations to test: 8

Configuration 1:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: False
dropout: 0.05

Configuration 2:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: False
dropout: 0.1

Configuration 3:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: True
dropout: 0.05

Configuration 4:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: True
dropout: 0.1

Configuration 5:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.05

Configuration 6:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.1

Configuration 7:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: True
dropout: 0.05

Configuration 8:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: True
dropout: 0.1


In [6]:
def run_experiment(config, exp_dir, exp_name):
    """Run a single experiment with given configuration"""
    # Set seeds for reproducibility
    set_seed(42)
    
    # Create data loaders
    train_loader, val_loader, scaler, asset_idx = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32,
        val_batch_size=32, 
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Initialize model
    model = InformerForecaster(config, asset_index=asset_idx)
    model.apply(_init_weights)
    learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Training configuration
    tcfg = TrainConfig(
        learning_rate=1e-4,
        weight_decay=0.01,
        max_steps=10000,
        warmup_steps=200,
        use_amp=True,
        device="cuda",
        patience=15,
        min_delta=0.0001
    )
    
    # Train model
    model, train_hist, val_hist, steps_hist, best_val_loss, best_val_step = train_model(
        model, train_loader, val_loader, tcfg, asset_index=asset_idx
    )
    
    # Best validation step
    best_step = best_val_step if best_val_step is not None else (min(val_hist, key=lambda t: t[1])[0] if val_hist else float('nan'))
    
    # Create validation loader for visualization
    _, val_loader_1, _, _ = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32, 
        val_batch_size=1,
        val_shuffle=True,
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Create experiment-specific directory
    exp_subdir = os.path.join(exp_dir, exp_name)
    os.makedirs(exp_subdir, exist_ok=True)
    
    # Save model and configuration
    model_save_path = os.path.join(exp_subdir, 'model.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'total_params': learnable_params,
        'train_hist': train_hist,
        'val_hist': val_hist,
        'steps_hist': steps_hist
    }, model_save_path)
    
    # Create and save plots
    fig, df_loss = create_loss_plot(
        train_hist, 
        val_hist, 
        steps_hist,
        total_params=learnable_params
    )
    fig.write_html(os.path.join(exp_subdir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_subdir, 'loss_history.csv'))
    
    # Calculate and save metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=10,
        save_dir=exp_subdir
    )
    
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_subdir, 'metrics.csv'), index=False)
    
    # Save summary metrics
    summary_metrics = metrics_df.mean().round(4)
    
    # Final train and best validation
    final_train_loss = train_hist[-1]
    best_val_loss = best_val_loss if best_val_loss is not None else (min(val_hist, key=lambda t: t[1])[1] if val_hist else float('nan'))
    
    return {
        'Experiment no': exp_name,
        'd_model': config['d_model'],
        'd_ff': config['d_ff'],
        'distill': config['distill'],
        'time embedding': config['use_time_embedding'],
        'dropout': config['dropout'],
        'learnable params': learnable_params,
        'Best Val Step': best_step,
        'train loss': final_train_loss,
        'best validation loss': best_val_loss,
        'jaggedness_ratio (pred/real)': float(summary_metrics['jaggedness_ratio']),
        'MSE': float(summary_metrics['mse']),
        'MAE': float(summary_metrics['mae'])
    }

In [7]:
# Create main experiments directory with timestamp
exp_dir = f"experiments_grid_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(exp_dir, exist_ok=True)

# Run all experiments
results = []
for i, config in enumerate(configs):
    print(f"\nRunning experiment {i+1}/{len(configs)}")
    print("Configuration:", config)
    
    # Create experiment name from parameters
    exp_name = (f"d{config['d_model']}_"
               f"{'dist' if config['distill'] else 'nodist'}_"
               f"{'time' if config['use_time_embedding'] else 'notime'}_"
               f"drop{config['dropout']}")
    
    # Run experiment
    try:
        result = run_experiment(config, exp_dir, exp_name)
        results.append(result)
        print(f"Experiment {exp_name} completed successfully")
        print(f"Best val loss: {result['best validation loss']:.6f}")
        print(f"MSE: {result['MSE']:.6f}")
    except Exception as e:
        print(f"Experiment {exp_name} failed with error: {str(e)}")
        continue

# Create results summary with specific column order
columns = [
    'Experiment no', 
    # Model Params
    'd_model', 'd_ff', 'distill', 'time embedding', 'dropout', 'learnable params',
    # Results
    'Best Val Step', 'train loss', 'best validation loss', 
    'jaggedness_ratio (pred/real)', 'MSE', 'MAE'
]

results_df = pd.DataFrame(results)[columns]

# Save results with proper formatting
results_df.to_csv(os.path.join(exp_dir, 'all_results.csv'), index=False, float_format='%.6f')

# Create summary visualizations
fig = px.scatter(results_df, 
                 x='best validation loss', 
                 y='MSE',
                 hover_data=columns,
                 title='Validation Loss vs MSE across experiments',
                 labels={'Experiment no': 'Experiment Name'})  # Update label
fig.write_html(os.path.join(exp_dir, 'results_scatter.html'))

# Save experiment configuration summary
config_summary = {
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'total_experiments': len(configs),
    'parameter_grid': param_grid,
    'base_config': {k: v for k, v in configs[0].items() if k not in param_grid},
    'experiment_names': [r['Experiment no'] for r in results]
}
with open(os.path.join(exp_dir, 'experiment_config.json'), 'w') as f:
    json.dump(config_summary, f, indent=4)

# Print best models by different metrics
print("\nBest models by validation loss:")
print(results_df.nsmallest(3, 'best validation loss')[columns])

print("\nBest models by MSE:")
print(results_df.nsmallest(3, 'MSE')[columns])


Running experiment 1/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 32, 'distill': False, 'use_time_embedding': False, 'dropout': 0.05, 'd_ff': 128}


2025-10-23 10:53:06,629 | INFO | Using fused AdamW: True
2025-10-23 10:53:09,971 | INFO | step 0 | train loss 1.037078 | lr 0.000e+00 | tok/s 3110.2
2025-10-23 10:53:12,325 | INFO | step 0 | VALIDATION loss 1.350633 | best 1.350633 | patience 0/15
2025-10-23 10:53:12,907 | INFO | step 10 | train loss 0.818363 | lr 5.000e-06 | tok/s 1308.3
2025-10-23 10:53:13,491 | INFO | step 20 | train loss 1.278331 | lr 1.000e-05 | tok/s 6585.4
2025-10-23 10:53:14,072 | INFO | step 30 | train loss 0.839135 | lr 1.500e-05 | tok/s 6633.9
2025-10-23 10:53:14,644 | INFO | step 40 | train loss 1.246686 | lr 2.000e-05 | tok/s 6735.1
2025-10-23 10:53:15,216 | INFO | step 50 | train loss 1.105822 | lr 2.500e-05 | tok/s 6731.0
2025-10-23 10:53:15,780 | INFO | step 60 | train loss 1.266533 | lr 3.000e-05 | tok/s 6822.6
2025-10-23 10:53:16,375 | INFO | step 70 | train loss 0.898683 | lr 3.500e-05 | tok/s 6487.4
2025-10-23 10:53:16,939 | INFO | step 80 | train loss 0.780116 | lr 4.000e-05 | tok/s 6824.2
2025-10-

Experiment d32_nodist_notime_drop0.05 completed successfully
Best val loss: 0.010904
MSE: 0.007600

Running experiment 2/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 32, 'distill': False, 'use_time_embedding': False, 'dropout': 0.1, 'd_ff': 128}


2025-10-23 11:03:52,654 | INFO | step 0 | VALIDATION loss 1.350633 | best 1.350633 | patience 0/15
2025-10-23 11:03:53,211 | INFO | step 10 | train loss 0.818397 | lr 5.000e-06 | tok/s 1400.9
2025-10-23 11:03:53,761 | INFO | step 20 | train loss 1.278317 | lr 1.000e-05 | tok/s 7004.1
2025-10-23 11:03:54,323 | INFO | step 30 | train loss 0.839175 | lr 1.500e-05 | tok/s 6859.0
2025-10-23 11:03:54,894 | INFO | step 40 | train loss 1.246739 | lr 2.000e-05 | tok/s 6748.7
2025-10-23 11:03:55,445 | INFO | step 50 | train loss 1.105821 | lr 2.500e-05 | tok/s 6992.7
2025-10-23 11:03:55,994 | INFO | step 60 | train loss 1.266538 | lr 3.000e-05 | tok/s 6996.2
2025-10-23 11:03:56,573 | INFO | step 70 | train loss 0.898678 | lr 3.500e-05 | tok/s 6666.3
2025-10-23 11:03:57,144 | INFO | step 80 | train loss 0.780149 | lr 4.000e-05 | tok/s 6739.6
2025-10-23 11:03:57,703 | INFO | step 90 | train loss 0.844235 | lr 4.500e-05 | tok/s 6878.0
2025-10-23 11:03:58,263 | INFO | step 100 | train loss 1.037709 

Experiment d32_nodist_notime_drop0.1 completed successfully
Best val loss: 0.009406
MSE: 0.006900

Running experiment 3/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 32, 'distill': False, 'use_time_embedding': True, 'dropout': 0.05, 'd_ff': 128}


...

Experiment d64_nodist_notime_drop0.1 completed successfully
Best val loss: 0.002922
MSE: 0.004700

Running experiment 7/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': True, 'dropout': 0.05, 'd_ff': 256}


2025-10-23 11:43:15,357 | INFO | step 0 | VALIDATION loss 1.106213 | best 1.106213 | patience 0/15
2025-10-23 11:43:16,006 | INFO | step 10 | train loss 1.288056 | lr 5.000e-06 | tok/s 1202.1
2025-10-23 11:43:16,647 | INFO | step 20 | train loss 1.071481 | lr 1.000e-05 | tok/s 6010.3
2025-10-23 11:43:17,280 | INFO | step 30 | train loss 0.827080 | lr 1.500e-05 | tok/s 6076.4
2025-10-23 11:43:17,949 | INFO | step 40 | train loss 0.922849 | lr 2.000e-05 | tok/s 5773.0
2025-10-23 11:43:18,641 | INFO | step 50 | train loss 0.912525 | lr 2.500e-05 | tok/s 5562.6
2025-10-23 11:43:19,308 | INFO | step 60 | train loss 0.727710 | lr 3.000e-05 | tok/s 5768.6
2025-10-23 11:43:19,960 | INFO | step 70 | train loss 0.918348 | lr 3.500e-05 | tok/s 5920.5
2025-10-23 11:43:20,606 | INFO | step 80 | train loss 0.811859 | lr 4.000e-05 | tok/s 5953.3
2025-10-23 11:43:21,255 | INFO | step 90 | train loss 0.916663 | lr 4.500e-05 | tok/s 5935.7
2025-10-23 11:43:21,931 | INFO | step 100 | train loss 0.858059 

Experiment d64_nodist_time_drop0.05 completed successfully
Best val loss: 0.006001
MSE: 0.005700

Running experiment 8/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': True, 'dropout': 0.1, 'd_ff': 256}


2025-10-23 11:54:45,190 | INFO | step 0 | VALIDATION loss 1.106213 | best 1.106213 | patience 0/15
2025-10-23 11:54:45,507 | INFO | step 10 | train loss 1.287938 | lr 5.000e-06 | tok/s 2808.2
2025-10-23 11:54:45,829 | INFO | step 20 | train loss 1.071500 | lr 1.000e-05 | tok/s 12009.4
2025-10-23 11:54:46,140 | INFO | step 30 | train loss 0.827169 | lr 1.500e-05 | tok/s 12397.2
2025-10-23 11:54:46,438 | INFO | step 40 | train loss 0.922786 | lr 2.000e-05 | tok/s 12901.3
2025-10-23 11:54:46,744 | INFO | step 50 | train loss 0.912459 | lr 2.500e-05 | tok/s 12647.3
2025-10-23 11:54:47,063 | INFO | step 60 | train loss 0.727614 | lr 3.000e-05 | tok/s 12107.5
2025-10-23 11:54:47,377 | INFO | step 70 | train loss 0.918407 | lr 3.500e-05 | tok/s 12384.7
2025-10-23 11:54:47,688 | INFO | step 80 | train loss 0.811752 | lr 4.000e-05 | tok/s 12423.7
2025-10-23 11:54:47,984 | INFO | step 90 | train loss 0.916676 | lr 4.500e-05 | tok/s 13064.1
2025-10-23 11:54:48,302 | INFO | step 100 | train loss 0

Experiment d64_nodist_time_drop0.1 completed successfully
Best val loss: 0.005291
MSE: 0.004700

Best models by validation loss:
                Experiment no  d_model  d_ff  distill  time embedding  \
4  d64_nodist_notime_drop0.05       64   256    False           False   
5   d64_nodist_notime_drop0.1       64   256    False           False   
7     d64_nodist_time_drop0.1       64   256    False            True   

   dropout  learnable params  Best Val Step  train loss  best validation loss  \
4     0.05            284929           4100    0.001597              0.002615   
5     0.10            284929           5100    0.001316              0.002922   
7     0.10            291033           5500    0.001625              0.005291   

   jaggedness_ratio (pred/real)     MSE     MAE  
4                        0.8014  0.0014  0.0292  
5                        0.8813  0.0047  0.0437  
7                        0.8317  0.0047  0.0527  

Best models by MSE:
                Experiment no  d